# Fase 4 - Modelagem

FP-Growth restrito a regras `contexto -> AltaGravidade`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import (
    ANOS_COMPLETOS,
    COLUNAS_CONTEXTO,
    DADOS_DIR,
    FIGURAS_DIR,
    MIN_SUPPORT,
    PROCESSED_DIR,
    TABELAS_DIR,
)

plt.rcParams["figure.figsize"] = (11, 5)
sns.set_style("whitegrid")

In [ ]:
from src.evaluation import traduzir_regras
from src.mining import (
    analise_sensibilidade,
    classificar_estabilidade_temporal,
    comparar_uso_solo,
    minerar_regras_alta_gravidade,
    regras_estabilidade_jan_abr,
    regras_estabilidade_principal,
)

df_onehot = pd.read_pickle(PROCESSED_DIR / "transacional.pkl")
df_meta = pd.read_pickle(PROCESSED_DIR / "transacional_meta.pkl")
itemsets, rules_all, rules_alta = minerar_regras_alta_gravidade(df_onehot, min_support=MIN_SUPPORT)
rules_alta = traduzir_regras(rules_alta)
print(f"Itemsets: {len(itemsets):,} | Regras alta gravidade: {len(rules_alta):,}")
display(rules_alta[["antecedents_str", "support", "confidence", "lift", "explicacao_natural"]].head(15))

In [ ]:
rules_por_ano = regras_estabilidade_principal(df_onehot, df_meta)
estabilidade = classificar_estabilidade_temporal(rules_por_ano)
print(estabilidade["status"].value_counts())
display(estabilidade.head(12))

In [ ]:
for estrato, rules in comparar_uso_solo(df_onehot, df_meta).items():
    print(estrato, len(rules))
    display(rules[["antecedents_str", "confidence", "lift"]].head(5))